# Histopathology Data Acquisition — TCGA-LUAD Diagnostic Slides (GDC)

Scaffold notebook, per `claude plans/as-of-now-we-refactored-tower.md` Phase 4.1.

TCGA LUAD whole-slide images are NOT part of the existing `luad_tcga_pan_can_atlas_2018/`
cBioPortal download — they must be pulled separately from the GDC Data Portal.

Steps:
1. Build a GDC cart/manifest filtered to `project_id=TCGA-LUAD`, `data_type="Slide Image"`,
   **Diagnostic Slides only** (barcode suffix `-DX1`/`-DX2`, not `-TS` frozen sections).
2. Download via `gdc-client download -m gdc_manifest.txt`.
3. Build `slide_manifest.csv` mapping `slide_id -> PATIENT_ID -> local .svs path`.
4. Report patient overlap against Expression/Mutation/Clinical immediately — this number
   gates the Phase 5 cohort decision.

Raw `.svs` files are stored outside git tracking at `Data/raw_data/histopathology/`
(already added to `.gitignore`).

In [ ]:
import os
import pandas as pd

RAW_DIR = "../Data/raw_data/histopathology"
MANIFEST_PATH = os.path.join(RAW_DIR, "gdc_manifest.txt")
SLIDE_MANIFEST_PATH = os.path.join(RAW_DIR, "slide_manifest.csv")

os.makedirs(RAW_DIR, exist_ok=True)

## Step 1 — GDC Manifest & Batched Download

Manifest is fetched programmatically:
```bash
uv run python EXP/fetch_gdc_manifest.py
```

Download **50 slides at a time** with verification:
```bash
uv run python EXP/download_gdc_slides.py --batch-size 50 --batch-index 0
uv run python EXP/verify_downloads.py --batch-size 50 --batch-index 0 --openslide-check
# Repeat for batch-index 1 … 9 after each batch passes verification
```

Progress: `download_status.json`, `download.log`. Re-run a batch to retry failed files only.

In [ ]:
# Manifest can be fetched programmatically (no GDC portal cart needed):
#   uv run python EXP/fetch_gdc_manifest.py
# Download slides via GDC Data API:
#   uv run python EXP/download_gdc_slides.py
# Or with gdc-client if installed:
#   gdc-client download -m Data/raw_data/histopathology/gdc_manifest.txt -d Data/raw_data/histopathology

import os
assert os.path.exists(MANIFEST_PATH), (
    f"Manifest not found at {MANIFEST_PATH} — run: uv run python EXP/fetch_gdc_manifest.py"
)

## Step 2 — Build `slide_manifest.csv`

Map each downloaded `.svs` file to its `PATIENT_ID` (first 12 characters of the
TCGA barcode, matching the format already used by `selected_clinical.csv` /
`selected_expression.csv` / `selected_mutation.csv` — no extra barcode wrangling
needed, confirmed during planning).

In [ ]:
def build_slide_manifest(raw_dir: str) -> pd.DataFrame:
    """Scan raw_dir for downloaded .svs files and map slide -> PATIENT_ID."""
    records = []
    for root, _, files in os.walk(raw_dir):
        for fname in files:
            if fname.lower().endswith(".svs"):
                # GDC diagnostic-slide filenames embed the full TCGA barcode,
                # e.g. TCGA-XX-XXXX-01Z-00-DX1....svs
                barcode = fname.split("-01Z")[0].split("-11Z")[0]
                patient_id = barcode[:12]
                records.append(
                    {
                        "slide_id": fname,
                        "PATIENT_ID": patient_id,
                        "path": os.path.join(root, fname),
                    }
                )
    return pd.DataFrame(records)


# slide_manifest = build_slide_manifest(RAW_DIR)
# slide_manifest.to_csv(SLIDE_MANIFEST_PATH, index=False)
# slide_manifest.head()

## Step 3 — Patient Overlap Report

Report `len(patients_with_histopathology ∩ patients_with_{expression,mutation,clinical})`
before doing any further Histopathology work — this drives the Phase 5 cohort decision
(see plan Phase 5.1).

In [ ]:
def report_patient_overlap(slide_manifest: pd.DataFrame) -> None:
    expr = pd.read_csv("../Data/processed_data/selected_expression.csv")["Patient_ID"]
    mut = pd.read_csv("../Data/processed_data/selected_mutation.csv")["Patient_ID"]
    clinical = pd.read_csv(
        "../Data/processed_data/mutation_data_processed/selected_clinical.csv"
    )["PATIENT_ID"]

    histo_patients = set(slide_manifest["PATIENT_ID"])
    overlap = histo_patients & set(expr) & set(mut) & set(clinical)

    print(f"Histopathology patients:        {len(histo_patients)}")
    print(f"Expression ∩ Mutation ∩ Clinical (existing 450-cohort baseline)")
    print(f"4-way intersection (new cohort): {len(overlap)}")


# report_patient_overlap(slide_manifest)